In [ ]:
import sys
sys.path.insert(0, "..")

CONFIG = "../configs/supervised/resnet18_asymmetric.yaml"
SEED   = 42
DEVICE = "cuda"   # change to "cpu" if no GPU available

In [ ]:
from scripts.run_pipeline_debug import run

results = run(config_path=CONFIG, seed=SEED, device=DEVICE)
print(f"\nReturned {len(results)} result dicts.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for ax, r in zip(axes, results):
    crop_np = r["tensor"].squeeze(0).numpy()
    vmin, vmax = np.percentile(crop_np, [2, 98])
    ax.imshow(crop_np, cmap="gray", vmin=vmin, vmax=vmax, origin="upper")

    label_color = "limegreen" if r["derived_label"] == 1 else "tomato"
    mismatch_tag = " ⚠" if r["label_mismatch"] else ""
    title = (
        f"F{r['frame_no']}  {r['detector']}\n"
        f"peaks={r['n_peaks']}  label={r['derived_label']}{mismatch_tag}"
    )
    ax.set_title(title, fontsize=9, color=label_color)
    ax.axis("off")

plt.suptitle("Preprocessing Debug Run — 10 Frames (all detectors)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

rows = []
for r in results:
    rows.append({
        "frame":         r["frame_no"],
        "detector":      r["detector"],
        "file":          r["file"],
        "frame_idx":     r["frame_idx"],
        "true_label":    r["true_label"],
        "n_peaks":       r["n_peaks"],
        "mismatch":      "⚠" if r["label_mismatch"] else "✓",
        "crop":          r["crop_decision"].split("  ")[0],
        "derived_label": r["derived_label"],
        "rot90_k":       r["augment"]["rot90_k"],
        "flip":          ("H" if r["augment"]["flip_h"] else "-") + ("V" if r["augment"]["flip_v"] else "-"),
        "gcn_max":       round(r["gcn_stats"]["max"], 2),
        "lcn_std":       round(r["lcn_stats"]["std"], 3),
        "assemble_ms":   round(r["assembly_time_s"] * 1000, 1),
    })

df = pd.DataFrame(rows).set_index("frame")
pd.set_option("display.max_colwidth", 30)
display(df)